# Notebook 01 — Data Ingestion & Cleaning

Loads raw CPCB CSV files from `data/raw/` if present, otherwise generates
realistic synthetic AQI data for all 10 Odisha cities (2019-01-01 – 2023-12-31).

Outputs: `data/processed/unified_clean.csv`

In [1]:
import asyncio, sys
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())


In [2]:
import os
import sys
import warnings
import logging
import numpy as np
import pandas as pd
from pathlib import Path

# Add project root to path so src/ is importable
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import CITIES, DIWALI_DATES

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
log = logging.getLogger(__name__)
warnings.filterwarnings('ignore')

print('Project root:', PROJECT_ROOT)
print('Cities:', list(CITIES.keys()))

Project root: D:\projects\odisha-aqi-advisor\odisha-aqi-advisor
Cities: ['Jharsuguda', 'Angul', 'Talcher', 'Rourkela', 'Sambalpur', 'Bhubaneswar', 'Cuttack', 'Bhadrak', 'Ganjam', 'Koraput']


In [3]:
DATE_START = '2019-01-01'
DATE_END   = '2023-12-31'

RAW_DIR       = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROCESSED_DIR / 'unified_clean.csv'

# AQI valid range
AQI_MIN, AQI_MAX = 0, 500

# Pollutant valid ranges (µg/m³ or ppm)
POLLUTANT_RANGES = {
    'pm25': (0, 500),
    'pm10': (0, 600),
    'no2':  (0, 400),
    'so2':  (0, 800),
    'o3':   (0, 400),
    'co':   (0, 50),
}

POLLUTANT_COLS = list(POLLUTANT_RANGES.keys())
print('Output path:', OUTPUT_PATH)

Output path: D:\projects\odisha-aqi-advisor\odisha-aqi-advisor\data\processed\unified_clean.csv


## Step 1 — Load raw data or generate synthetic data

In [4]:
def generate_synthetic_data(seed: int = 42) -> pd.DataFrame:
    """Generate realistic synthetic AQI data for all 10 cities."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range(DATE_START, DATE_END, freq='D')

    # City-level base AQI config
    city_config = {
        # Industrial: high base, high variance
        'Jharsuguda':  {'base': 180, 'std': 60, 'tier': 'industrial'},
        'Angul':       {'base': 170, 'std': 55, 'tier': 'industrial'},
        'Talcher':     {'base': 190, 'std': 65, 'tier': 'industrial'},
        'Rourkela':    {'base': 160, 'std': 50, 'tier': 'industrial'},
        'Sambalpur':   {'base': 150, 'std': 50, 'tier': 'industrial'},
        # Urban: medium base
        'Bhubaneswar': {'base': 100, 'std': 35, 'tier': 'urban'},
        'Cuttack':     {'base': 110, 'std': 38, 'tier': 'urban'},
        'Bhadrak':     {'base': 95,  'std': 32, 'tier': 'urban'},
        # Clean baseline: low base
        'Ganjam':      {'base': 65,  'std': 22, 'tier': 'clean'},
        'Koraput':     {'base': 55,  'std': 18, 'tier': 'clean'},
    }

    diwali_ts = pd.to_datetime(DIWALI_DATES)
    rows = []

    for city, cfg in city_config.items():
        base = cfg['base']
        std  = cfg['std']

        for d in dates:
            month = d.month

            # Seasonal multiplier
            if month in (11, 12, 1, 2):   # winter — higher AQI
                seasonal = 1.30
            elif month in (7, 8, 9):       # monsoon — lower AQI
                seasonal = 0.70
            elif month in (3, 4, 5):       # summer — slightly elevated
                seasonal = 1.10
            else:
                seasonal = 1.00

            # Diwali spike
            diwali_spike = 1.0
            for dw in diwali_ts:
                if abs((d - dw).days) <= 3:
                    diwali_spike = 1.40
                    break

            aqi_mean = base * seasonal * diwali_spike
            aqi = float(rng.normal(aqi_mean, std))

            # Clip to valid range per tier
            if cfg['tier'] == 'industrial':
                aqi = np.clip(aqi, 80, 350)
            elif cfg['tier'] == 'urban':
                aqi = np.clip(aqi, 50, 200)
            else:
                aqi = np.clip(aqi, 30, 120)

            # Derive pollutants from AQI with realistic ratios + noise
            pm25 = float(np.clip(rng.normal(aqi * 0.45, aqi * 0.05), 0, 500))
            pm10 = float(np.clip(rng.normal(aqi * 0.70, aqi * 0.07), 0, 600))
            no2  = float(np.clip(rng.normal(aqi * 0.18, aqi * 0.03), 0, 400))
            so2  = float(np.clip(rng.normal(aqi * 0.12, aqi * 0.02), 0, 800))
            o3   = float(np.clip(rng.normal(aqi * 0.10, aqi * 0.02), 0, 400))
            co   = float(np.clip(rng.normal(aqi * 0.008, aqi * 0.001), 0, 50))

            rows.append({
                'city': city,
                'date': d,
                'aqi':  round(aqi, 2),
                'pm25': round(pm25, 2),
                'pm10': round(pm10, 2),
                'no2':  round(no2, 2),
                'so2':  round(so2, 2),
                'o3':   round(o3, 2),
                'co':   round(co, 2),
            })

    df = pd.DataFrame(rows)
    log.info('Synthetic data generated: %d rows for %d cities', len(df), df['city'].nunique())
    return df


In [5]:
def load_raw_csvs(raw_dir: Path) -> pd.DataFrame | None:
    """Attempt to load CSV files from data/raw/. Returns None if none found."""
    csv_files = list(raw_dir.glob('*.csv'))
    if not csv_files:
        return None
    frames = []
    for f in csv_files:
        try:
            frames.append(pd.read_csv(f))
            log.info('Loaded raw file: %s', f.name)
        except Exception as e:
            log.warning('Failed to load %s: %s', f.name, e)
    return pd.concat(frames, ignore_index=True) if frames else None


raw_df = load_raw_csvs(RAW_DIR)

if raw_df is not None:
    log.info('Using real raw data: %d rows', len(raw_df))
    df = raw_df.copy()
else:
    log.warning(
        'No CSV files found in data/raw/. '
        'Download real data from https://app.cpcbccr.com/ccr/ '
        'For now, generating synthetic data for development/demo purposes.'
    )
    df = generate_synthetic_data()

print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
df.head(3)

INFO: Synthetic data generated: 18260 rows for 10 cities


Loaded 18,260 rows, 9 columns


,city,date,aqi,pm25,pm10,no2,so2,o3,co
0,Jharsuguda,2019-01-01,252.28,100.41,189.85,52.53,20.43,18.66,2.05
1,Jharsuguda,2019-01-02,215.03,96.58,137.68,44.38,29.15,21.79,1.96
2,Jharsuguda,2019-01-03,262.05,106.66,190.20,39.63,36.05,25.94,2.05


## Cleaning Pipeline

In [6]:
# Step 1: Lowercase column names, strip whitespace, replace spaces with underscores
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(' ', '_', regex=False)
)
print('Columns after normalisation:', df.columns.tolist())

Columns after normalisation: ['city', 'date', 'aqi', 'pm25', 'pm10', 'no2', 'so2', 'o3', 'co']


In [7]:
# Step 2: Parse date column; drop rows where date is NaT
before = len(df)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])
dropped = before - len(df)
log.info('Step 2 — dropped %d rows with unparseable dates', dropped)
print(f'Rows after date parse: {len(df):,} (dropped {dropped})')

INFO: Step 2 — dropped 0 rows with unparseable dates


Rows after date parse: 18,260 (dropped 0)


In [8]:
# Step 3: Strip and title-case city column; map lowercase variants; drop unknown cities
CITY_NAMES = set(CITIES.keys())

# Build a lowercase → canonical mapping
city_map = {c.lower(): c for c in CITY_NAMES}

df['city'] = df['city'].astype(str).str.strip().str.title()
# Also try lowercase mapping for any remaining mismatches
df['city'] = df['city'].apply(lambda c: city_map.get(c.lower(), c))

before = len(df)
df = df[df['city'].isin(CITY_NAMES)]
dropped = before - len(df)
log.info('Step 3 — dropped %d rows with unknown city names', dropped)
print(f'Rows after city filter: {len(df):,} (dropped {dropped})')
print('Cities present:', sorted(df['city'].unique()))

INFO: Step 3 — dropped 0 rows with unknown city names


Rows after city filter: 18,260 (dropped 0)
Cities present: ['Angul', 'Bhadrak', 'Bhubaneswar', 'Cuttack', 'Ganjam', 'Jharsuguda', 'Koraput', 'Rourkela', 'Sambalpur', 'Talcher']


In [9]:
# Step 4: Cast pollutant columns to float64; non-castable → NaN
# Ensure all expected columns exist (add as NaN if missing)
for col in ['aqi'] + POLLUTANT_COLS:
    if col not in df.columns:
        df[col] = np.nan
        log.warning('Column %s missing from source data — filled with NaN', col)

for col in ['aqi'] + POLLUTANT_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('float64')

print('Dtypes after cast:')
print(df[['aqi'] + POLLUTANT_COLS].dtypes)

Dtypes after cast:
aqi     float64
pm25    float64
pm10    float64
no2     float64
so2     float64
o3      float64
co      float64
dtype: object


In [10]:
# Step 5: Range checks
# AQI outside 0-500 → drop row
before = len(df)
df = df[(df['aqi'] >= AQI_MIN) & (df['aqi'] <= AQI_MAX)]
dropped_aqi = before - len(df)
log.info('Step 5 — dropped %d rows with AQI outside [0, 500]', dropped_aqi)

# Pollutants outside range → set NaN (don't drop row)
for col, (lo, hi) in POLLUTANT_RANGES.items():
    if col in df.columns:
        mask = (df[col] < lo) | (df[col] > hi)
        n_out = mask.sum()
        if n_out:
            df.loc[mask, col] = np.nan
            log.info('Step 5 — set %d out-of-range values to NaN in %s', n_out, col)

print(f'Rows after AQI range check: {len(df):,} (dropped {dropped_aqi})')

INFO: Step 5 — dropped 0 rows with AQI outside [0, 500]


Rows after AQI range check: 18,260 (dropped 0)


In [11]:
# Step 6: Sort by city ascending, then date ascending
df = df.sort_values(['city', 'date']).reset_index(drop=True)
print('Sorted by city, date. Shape:', df.shape)

Sorted by city, date. Shape:

 (18260, 9)


In [12]:
# Step 7: Per city group — forward-fill NaN for gaps of 1-3 consecutive days
# Track which values were originally NaN (for data_quality column)
originally_nan_aqi = df['aqi'].isna()

def ffill_group(group):
    return group.ffill(limit=3)

pollutant_and_aqi = ['aqi'] + POLLUTANT_COLS
df[pollutant_and_aqi] = (
    df.groupby('city', group_keys=False)[pollutant_and_aqi]
    .apply(ffill_group)
)

ffilled_aqi = (~originally_nan_aqi) | df['aqi'].notna()
n_ffilled = (originally_nan_aqi & df['aqi'].notna()).sum()
log.info('Step 7 — forward-filled %d AQI values', n_ffilled)
print(f'Forward-filled {n_ffilled} AQI values')

INFO: Step 7 — forward-filled 0 AQI values


Forward-filled 0 AQI values


In [13]:
# Step 8: Per city group — fill remaining NaN with city-level median
still_nan_aqi = df['aqi'].isna()

def median_fill_group(group):
    for col in pollutant_and_aqi:
        med = group[col].median()
        group[col] = group[col].fillna(med)
    return group

df = df.groupby('city', group_keys=False).apply(median_fill_group)

n_median_filled = still_nan_aqi.sum()
log.info('Step 8 — median-filled %d AQI values', n_median_filled)
print(f'Median-filled {n_median_filled} AQI values')

INFO: Step 8 — median-filled 0 AQI values


Median-filled 0 AQI values


In [14]:
# Step 9: Add data_quality column
df['data_quality'] = 'original'
df.loc[originally_nan_aqi & df['aqi'].notna() & ~still_nan_aqi, 'data_quality'] = 'forward_filled'
df.loc[still_nan_aqi, 'data_quality'] = 'median_filled'

print('data_quality distribution:')
print(df['data_quality'].value_counts())

data_quality distribution:


data_quality
original    18260
Name: count, dtype: int64


In [15]:
# Step 10: Assert no NaN remains in aqi column
nan_count = df['aqi'].isna().sum()
assert nan_count == 0, f'FAILED: {nan_count} NaN values remain in aqi column!'
log.info('Step 10 — assertion passed: no NaN in aqi column')
print('✓ No NaN values in aqi column')

INFO: Step 10 — assertion passed: no NaN in aqi column


✓ No NaN values in aqi column


In [16]:
# Filter to date range 2019-01-01 to 2023-12-31 inclusive
before = len(df)
df = df[(df['date'] >= DATE_START) & (df['date'] <= DATE_END)]
dropped = before - len(df)
log.info('Date range filter — dropped %d out-of-range rows', dropped)
print(f'Rows after date range filter: {len(df):,} (dropped {dropped})')

INFO: Date range filter — dropped 0 out-of-range rows


Rows after date range filter: 18,260 (dropped 0)


In [17]:
# Deduplicate (city, date) pairs — keep first occurrence
before = len(df)
df = df.drop_duplicates(subset=['city', 'date'], keep='first')
dropped = before - len(df)
log.info('Deduplication — dropped %d duplicate (city, date) rows', dropped)
print(f'Rows after deduplication: {len(df):,} (dropped {dropped})')

INFO: Deduplication — dropped 0 duplicate (city, date) rows


Rows after deduplication: 18,260 (dropped 0)


## Step 11 — Save to `data/processed/unified_clean.csv`

In [18]:
# Select final schema columns
SCHEMA_COLS = ['city', 'date', 'aqi', 'pm25', 'pm10', 'no2', 'so2', 'o3', 'co', 'data_quality']

# Ensure all schema columns exist
for col in SCHEMA_COLS:
    if col not in df.columns:
        df[col] = np.nan

df_out = df[SCHEMA_COLS].copy()

# Format date as ISO string
df_out['date'] = df_out['date'].dt.strftime('%Y-%m-%d')

df_out.to_csv(OUTPUT_PATH, index=False)
log.info('Saved unified_clean.csv: %d rows, %d columns', len(df_out), len(SCHEMA_COLS))
print(f'\n✓ Saved to {OUTPUT_PATH}')
print(f'  Shape: {df_out.shape}')
print(f'  Columns: {df_out.columns.tolist()}')
print(f'  Date range: {df_out["date"].min()} → {df_out["date"].max()}')
print(f'  Cities: {sorted(df_out["city"].unique())}')
df_out.head()

INFO: Saved unified_clean.csv: 18260 rows, 10 columns



✓ Saved to D:\projects\odisha-aqi-advisor\odisha-aqi-advisor\data\processed\unified_clean.csv
  Shape: (18260, 10)
  Columns: ['city', 'date', 'aqi', 'pm25', 'pm10', 'no2', 'so2', 'o3', 'co', 'data_quality']
  Date range: 2019-01-01 → 2023-12-31
  Cities: ['Angul', 'Bhadrak', 'Bhubaneswar', 'Cuttack', 'Ganjam', 'Jharsuguda', 'Koraput', 'Rourkela', 'Sambalpur', 'Talcher']


,city,date,aqi,pm25,pm10,no2,so2,o3,co,data_quality
0,Angul,2019-01-01,350.00,151.11,205.62,48.15,52.32,34.46,2.86,original
1,Angul,2019-01-02,191.16,81.70,123.28,37.11,22.57,20.78,1.59,original
2,Angul,2019-01-03,197.58,74.50,116.92,53.50,29.55,25.05,1.45,original
3,Angul,2019-01-04,221.52,90.65,134.72,35.96,32.46,25.88,1.56,original
4,Angul,2019-01-05,203.69,99.29,134.44,37.15,26.08,21.96,1.68,original


## Validation Summary

In [19]:
# Quick validation of the output file
df_check = pd.read_csv(OUTPUT_PATH, parse_dates=['date'])

print('=== unified_clean.csv validation ===')
print(f'Rows:    {len(df_check):,}')
print(f'Columns: {df_check.columns.tolist()}')
print(f'Cities:  {sorted(df_check["city"].unique())}')
print(f'Date range: {df_check["date"].min().date()} → {df_check["date"].max().date()}')

# Check no NaN in aqi
assert df_check['aqi'].isna().sum() == 0, 'NaN found in aqi!'
print('✓ No NaN in aqi')

# Check no duplicate (city, date)
dupes = df_check.duplicated(subset=['city', 'date']).sum()
assert dupes == 0, f'{dupes} duplicate (city, date) pairs found!'
print('✓ No duplicate (city, date) pairs')

# Check date range
assert df_check['date'].min() >= pd.Timestamp('2019-01-01'), 'Date before 2019-01-01 found!'
assert df_check['date'].max() <= pd.Timestamp('2023-12-31'), 'Date after 2023-12-31 found!'
print('✓ All dates within 2019-01-01 to 2023-12-31')

# Check all 10 cities present
assert set(df_check['city'].unique()) == set(CITIES.keys()), 'Missing cities!'
print('✓ All 10 cities present')

print('\n=== All validations passed ===')

=== unified_clean.csv validation ===
Rows:    18,260
Columns: ['city', 'date', 'aqi', 'pm25', 'pm10', 'no2', 'so2', 'o3', 'co', 'data_quality']
Cities:  ['Angul', 'Bhadrak', 'Bhubaneswar', 'Cuttack', 'Ganjam', 'Jharsuguda', 'Koraput', 'Rourkela', 'Sambalpur', 'Talcher']
Date range: 2019-01-01 → 2023-12-31
✓ No NaN in aqi
✓ No duplicate (city, date) pairs
✓ All dates within 2019-01-01 to 2023-12-31
✓ All 10 cities present

=== All validations passed ===
